# Inspect FASERCal NPZ events

This notebook verifies the raw `.npz` structure before any model code is used. It focuses on `reco_hits`, expected to contain sparse 3DCal voxel hits with `[x, y, z, module, energy, ...]` columns.

In [ ]:
from pathlib import Path
import sys

PROJECT = Path('/home/fcufino/FabioFaserDL/projects/Nu-JEPA')
sys.path.insert(0, str(PROJECT))

from src.data.faser_npz_dataset import list_npz_files
from src.data.sparse_preprocessing import inspect_npz, extract_reco_hits, hits_to_sparse_event, summarize_active_patches
from src.visualization.plotly_3dcal import scatter_event

DATASET = Path('/scratch/fcufino/events_v8.0_1000')
DETECTOR_SIZE = (48, 48, 200)
PATCH_SIZE = (12, 12, 10)
files = list_npz_files(str(DATASET))
print(f'Found {len(files)} npz files under {DATASET}')
if not files:
    raise FileNotFoundError(f'No .npz files found under {DATASET}')
files[:5]

In [ ]:
summaries = [inspect_npz(path) for path in files[:3]]
for summary in summaries:
    print('\n', summary['path'])
    print('keys:', summary['keys'])
    print('reco_hits:', summary.get('reco_hits_summary'))

In [ ]:
import numpy as np

path = files[0]
with np.load(path, allow_pickle=True) as npz:
    hits = extract_reco_hits(npz)
event = hits_to_sparse_event(hits, detector_size=DETECTOR_SIZE, coord_mode='discrete', path=path)
patches = summarize_active_patches(event, detector_size=DETECTOR_SIZE, patch_size=PATCH_SIZE)
print('raw reco_hits shape:', hits.shape)
print('active voxels:', event.coords.shape[0])
print('active patches:', patches.patch_ids.numel())
print('energy sum:', float(event.energy.sum()))
print('coord min:', event.coords.min(dim=0).values.tolist() if event.coords.numel() else None)
print('coord max:', event.coords.max(dim=0).values.tolist() if event.coords.numel() else None)

In [ ]:
fig = scatter_event(event, title='Full 3DCal reco_hits')
fig.show()

## Edge occupancy check for 3D reco_hits

This checks whether the 3D `reco_hits` actually occupy the full declared `(48, 48, 200)` detector grid. If the detector grid is full size, valid X/Y bins could go up to `47`; the current v8 files only fill up to `42`.


In [ ]:
import numpy as np
from pathlib import Path

def scan_reco_hit_edges(files, edge_xyz=(43, 43, 200)):
    mins = np.array([np.inf, np.inf, np.inf], dtype=np.float64)
    maxs = np.array([-np.inf, -np.inf, -np.inf], dtype=np.float64)
    edge_counts = np.zeros(3, dtype=np.int64)
    total_hits = 0
    files_with_edge_hits = []

    edge_xyz = np.asarray(edge_xyz, dtype=np.float64)
    for path in files:
        with np.load(path, allow_pickle=True) as npz:
            if "reco_hits" not in npz:
                continue
            hits = extract_reco_hits(npz)
        if hits.size == 0:
            continue

        coords = hits[:, :3]
        mins = np.minimum(mins, coords.min(axis=0))
        maxs = np.maximum(maxs, coords.max(axis=0))
        hits_past_edge = coords >= edge_xyz
        edge_counts += hits_past_edge.sum(axis=0).astype(np.int64)
        if hits_past_edge.any():
            files_with_edge_hits.append(Path(path).name)
        total_hits += len(coords)

    print(f"files checked: {len(files)}")
    print(f"total reco hits: {total_hits}")
    print(f"coordinate min [x,y,z]: {mins.tolist()}")
    print(f"coordinate max [x,y,z]: {maxs.tolist()}")
    print(f"hits with coord >= {tuple(edge_xyz.astype(int))}: {edge_counts.tolist()}")
    print(f"files with any hit in those edge regions: {len(files_with_edge_hits)}")

    return {
        "coord_min": mins,
        "coord_max": maxs,
        "edge_counts": edge_counts,
        "files_with_edge_hits": files_with_edge_hits,
    }


edge_summary = scan_reco_hit_edges(files)
